# Symbolic Math AI: Training → ONNX (CUDA) → Streamlit App

This end-to-end notebook fine-tunes the Symbolic Math AI on preprocessed datasets, exports the trained model to ONNX optimized for CUDA, and shows how to launch the Streamlit app with Tree-of-Thoughts (ToT), SymPy, and SHAP.

Prereqs:
- Ensure `requirements.txt` is installed
- Set `HUGGING_FACE_HUB_TOKEN` if using gated models (we'll use a small, open causal model in this notebook)

Outputs:
- Trained model in `./trained_model_custom`
- ONNX model in `./onnx_model_custom`


In [1]:
import os
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from symbolic_math_ai import TrainingConfig, MathModelTrainer

# Use your HF token if needed (set in kernel env)
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.getenv("HUGGING_FACE_HUB_TOKEN", "")

# Math-focused small model; good starting point
custom_output_dir = "./trained_math_model_qwen"
model_name = "Qwen/Qwen2.5-Math-1.5B"

config = TrainingConfig(
    model_name=model_name,
    output_dir=custom_output_dir,
    batch_size=1,                   # safer default
    learning_rate=2e-5,
    num_epochs=2,
    max_length=512,
    max_samples_per_dataset=1000,   # scale later
    eval_steps=200,
    save_steps=200,
    use_quantization=False,
    use_gradient_checkpointing=True,
    tot_max_depth=3,
    tot_max_children=3,
)

trainer = MathModelTrainer(config)

model = trainer.train_model()
eval_results = trainer.evaluate_model(model)
print("Evaluation:", eval_results)


2025-08-15 14:13:28,802 - WARNING - Hugging Face token not found. Model loading may fail.
2025-08-15 14:13:28,805 - WARNING - Hugging Face token not found. Model loading may fail.
2025-08-15 14:13:28,806 - INFO - Initializing Math Model Trainer...
2025-08-15 14:13:28,807 - INFO - Starting model training with Trainer...
2025-08-15 14:13:28,809 - INFO - Loading model: Qwen/Qwen2.5-Math-1.5B
2025-08-15 14:13:40,812 - INFO - Model and tokenizer loaded successfully
2025-08-15 14:13:40,814 - INFO - Loading preprocessed datasets...
2025-08-15 14:13:40,957 - INFO - Loaded 1000 samples from preprocessed_gsm8k_train.csv
2025-08-15 14:13:41,438 - INFO - Loaded 1000 samples from preprocessed_mathqa_train.csv
2025-08-15 14:13:41,457 - INFO - Loaded 700 samples from preprocessed_svamp_train.csv
2025-08-15 14:13:41,465 - INFO - Loaded 400 samples from preprocessed_math500_train.csv
2025-08-15 14:13:41,466 - INFO - Total training samples: 2480
2025-08-15 14:13:41,468 - INFO - Total validation samples:

: 

In [ ]:
# Environment check
import torch, onnxruntime as ort, transformers
print("cuda:", torch.cuda.is_available())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("ort:", ort.__version__, "providers:", ort.get_available_providers())


cuda: False
torch: 2.8.0+cpu
transformers: 4.53.3
ort: 1.17.0 providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [ ]:
# Exact Match evaluation helper (simple numeric/string EM)
import re, numpy as np

def extract_final_answer(text: str):
    m = re.findall(r"[-+]?[0-9]*\.?[0-9]+", text)
    return m[-1] if m else text.strip().lower()

def generate_and_em(trainer, dataset, n=200):
    model, tok = trainer.model, trainer.data_collator.tokenizer
    em_count, total = 0, 0
    model.eval()
    for i in range(min(n, len(dataset))):
        item = dataset[i]
        inp_ids = item["input_ids"].unsqueeze(0).to(model.device)
        attn = item["attention_mask"].unsqueeze(0).to(model.device)
        with torch.no_grad():
            gen = model.generate(
                input_ids=inp_ids,
                attention_mask=attn,
                max_new_tokens=128,
                do_sample=False,
                num_beams=1,
                pad_token_id=tok.eos_token_id,
            )
        out = tok.decode(gen[0], skip_special_tokens=True)
        pred = extract_final_answer(out)
        gold_ids = item["labels"]
        gold_text = tok.decode(gold_ids[gold_ids != -100], skip_special_tokens=True)
        gold = extract_final_answer(gold_text)
        em_count += int(pred == gold)
        total += 1
    return {"exact_match": em_count / max(total, 1)}


In [ ]:
# After training, you can compute EM on a subset of validation
_, val_dataset = trainer.prepare_datasets()
em = generate_and_em(trainer, val_dataset, n=200)
print(em)


In [ ]:
# (Optional) ONNX export step was removed for stability on this machine.
# You can re-enable later by restoring onnx_export.py and installing Optimum/ORT.
print("Skipping ONNX export. Using PyTorch model at:", custom_output_dir)


## Launch the Streamlit app (PyTorch backend)

Use from a terminal (stop with Ctrl+C):

```bash
streamlit run app_streamlit.py -- --model_dir ./trained_math_model_qwen
```

In the app sidebar:
- Backend: HF (PyTorch)
- Optional SHAP (slower)
- Adjust ToT depth/children as needed
